## **Human Activity Recognition**

**Mount Google Drive with the Google Colab Notebook**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd "/content/drive/MyDrive/yolo26"

In [ ]:
! mkdir -p "Human_Activity_Recognition"

In [ ]:
%cd "/content/drive/MyDrive/yolo26/Human_Activity_Recognition"

**Install All the Required Packages**

In [ ]:
! pip install ultralytics

**Import All the Required Libraries**

In [ ]:
import ultralytics
from IPython.display import Image

**Check Ultralytics Version and Setup Completion**

In [ ]:
ultralytics.checks()

**Download the Dataset from Roboflow**

In [ ]:
! pip install roboflow

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="xxxxxxxxxxxxxxxxxxxxxx")
project = rf.workspace("pose-detection-twxbg").project("human-activity-ce7zu")
version = project.version(2)
dataset = version.download("yolov8")

In [ ]:
# *** Change path in data.yaml file for Google Colab ***
# path: /content/dataset/.....
# test: test/images    # relative to 'path'
# train: train/images  # relative to 'path'
# val: valid/images    # relative to 'path'

**Train / Fine-Tune YOLO26 Pose Estimation Model on Human Activity Recognition Dataset**

In [ ]:
dataset.location

In [ ]:
# Start training from a pretrained *.pt model
! yolo task=pose mode=train model="yolo26n-pose.pt" data="Human-activity-2/data.yaml" epochs=100 imgsz=640

**Validate the Fine-Tune Model**

In [ ]:
! yolo task=detect mode=val model="runs/pose/train/weights/best.pt" data="Human-activity-2/data.yaml"

#**Examine the Training Results**

### **Confusion Matrix**

In [ ]:
from IPython.display import Image, display
Image("runs/pose/train/confusion_matrix.png", width = 800)

### **Percision - Confidence Curve**

Precision = TP/(TP + FP)

Precision in Computer Vision is a Metric that tells you:

Out of all detections your model predicted as positive, how many were actually correct?

TP (True Positives) -> Correct Detections

FP (False Positive) -> Wrong Detection (Model predicted an object but it wasn't actually there)


A high Precision means:

*   Few false alarms
*   Most detected objects are correct
*   The model is reliable when it says “I found something”

A low Precision means:

*   Many false positives
*   Model keeps detecting objects where none exist

8 detections are correct (TP = 8)

2 detections are wrong (FP = 2)

In [ ]:
Image("runs/pose/train/PoseP_curve.png", width = 800)

### **Recall - Confidence Curve**

Recall measures how well your model finds all the relevant objects.


Recall tells you: Out of all the actual objects present, how many did the model detect?

Recall = TP / (TP + FN)


Where:

TP (True Positives) → Correct detections

FN (False Negatives) → Objects your model missed


Correctly detected 8 (TP = 8)

Missed 2 (FN = 2)

In [ ]:
Image("runs/pose/train/PoseR_curve.png", width = 800)

### **Model Predictions on Validation Batch**

In [ ]:
Image("runs/pose/train/val_batch1_pred.jpg", width = 800)

In [ ]:
Image("runs/pose/train/val_batch2_pred.jpg", width = 800)

### **Inference on Test Dataset Images using the Trained/ Fine-Tune Pose Estimation Model**

In [ ]:
! gdown "https://drive.google.com/uc?id=1a9A9jIDBNv3qeHOasMrjnXLhFylrE-tj&confirm=t"

In [ ]:
! yolo task=pose mode=predict model="best.pt" conf=0.25 source="Human-activity-2/test/images" save=True

**Display the Output Images**

In [ ]:
import glob
import os
from IPython.display import Image as IPyImage, display

latest_folder = max(glob.glob('runs/pose/predict/'), key=os.path.getmtime)

images = glob.glob(f'{latest_folder}/*.jpg')

selected_images = images[80:88] + images[120:130] + images[140:148]

for img in selected_images:
    display(IPyImage(filename=img, width=600))
    print("\n")


### **Inference on a Random Video**

In [ ]:
! gdown "https://drive.google.com/uc?id=19jREqMUWoJgixJgwgeHvvsPu82neJITp&confirm=t"

In [ ]:
! yolo task=pose mode=predict model="best.pt" conf=0.25 source="video1.mp4" save=True

In [ ]:
! rm "/content/result_compressed.mp4"

In [ ]:
from IPython.display import HTML
from base64 import b64encode
import os

# Input video path
save_path = 'runs/pose/predict7/video1.avi'

# Compressed video path
compressed_path = "/content/result_compressed.mp4"

os.system(f"ffmpeg -i {save_path} -vcodec libx264 {compressed_path}")

# Show video
mp4 = open(compressed_path,'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML("""
<video width=400 controls>
      <source src="%s" type="video/mp4">
</video>
""" % data_url)